"""
NFL Field Coordinate Transformation System
Converts YOLO detections (yard numbers, hashes, players) into top-down field coordinates
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.cluster import DBSCAN
from typing import List, Dict, Tuple, Optional

# ============================================================================
# STEP 1: IDENTIFY BASELINE YARD LINE
# ============================================================================

def find_baseline_yard_line(yard_detections: List[Dict]) -> Optional[Tuple[int, List[Dict]]]:
    """
    Find the best baseline yard line from detected yard numbers.
    
    Args:
        yard_detections: List of dicts with keys:
            - 'class_name': e.g., 'yard_40'
            - 'confidence': float
            - 'bbox': (x1, y1, x2, y2)
            - 'yard_number_pred': int (e.g., 40)
            - 'yard_number_conf': float
    
    Returns:
        (yard_value, [left_detection, right_detection]) or None
    """
    # Group detections by predicted yard number
    yard_groups = defaultdict(list)
    for det in yard_detections:
        yard_val = det.get('yard_number_pred')
        if yard_val is not None:
            yard_groups[yard_val].append(det)
    
    # For each yard value, find the two highest-confidence detections
    best_baseline = None
    best_score = -1
    
    for yard_val, dets in yard_groups.items():
        if len(dets) < 2:
            continue
        
        # Sort by confidence
        sorted_dets = sorted(dets, key=lambda x: x.get('yard_number_conf', x['confidence']), reverse=True)
        
        # Take top 2
        top_two = sorted_dets[:2]
        
        # Check if they're on opposite sides (left vs right of image)
        bbox1 = top_two[0]['bbox']
        bbox2 = top_two[1]['bbox']
        
        center1_x = (bbox1[0] + bbox1[2]) / 2
        center2_x = (bbox2[0] + bbox2[2]) / 2
        
        # They should be separated horizontally
        x_separation = abs(center1_x - center2_x)
        
        # Score based on combined confidence and separation
        combined_conf = top_two[0].get('yard_number_conf', top_two[0]['confidence']) + \
                       top_two[1].get('yard_number_conf', top_two[1]['confidence'])
        score = combined_conf * (x_separation / 1000)  # Normalize by typical image width
        
        if score > best_score:
            best_score = score
            # Order left to right
            if center1_x < center2_x:
                best_baseline = (yard_val, [top_two[0], top_two[1]])
            else:
                best_baseline = (yard_val, [top_two[1], top_two[0]])
    
    return best_baseline


def get_baseline_points(left_det: Dict, right_det: Dict) -> Tuple[np.ndarray, np.ndarray]:
    """Get center points of the two baseline yard number detections."""
    bbox_left = left_det['bbox']
    bbox_right = right_det['bbox']
    
    left_point = np.array([
        (bbox_left[0] + bbox_left[2]) / 2,
        (bbox_left[1] + bbox_left[3]) / 2
    ])
    
    right_point = np.array([
        (bbox_right[0] + bbox_right[2]) / 2,
        (bbox_right[1] + bbox_right[3]) / 2
    ])
    
    return left_point, right_point


# ============================================================================
# STEP 2: IDENTIFY HASH ROWS
# ============================================================================

def cluster_hashes_into_rows(hash_detections: List[Dict], y_tolerance: float = 20) -> List[List[Dict]]:
    """
    Cluster hash marks into horizontal rows based on y-coordinate alignment.
    
    Args:
        hash_detections: List of hash mark detections
        y_tolerance: Maximum y-distance for hashes to be in same row
    
    Returns:
        List of hash rows, each row is a list of detections
    """
    if not hash_detections:
        return []
    
    # Get center points
    centers = []
    for det in hash_detections:
        bbox = det['bbox']
        center_y = (bbox[1] + bbox[3]) / 2
        centers.append(center_y)
    
    centers = np.array(centers).reshape(-1, 1)
    
    # Cluster by y-coordinate
    clustering = DBSCAN(eps=y_tolerance, min_samples=2).fit(centers)
    labels = clustering.labels_
    
    # Group into rows
    rows = defaultdict(list)
    for idx, label in enumerate(labels):
        if label != -1:  # Ignore noise
            rows[label].append(hash_detections[idx])
    
    # Sort each row by x-coordinate (left to right)
    sorted_rows = []
    for row_dets in rows.values():
        row_dets.sort(key=lambda d: (d['bbox'][0] + d['bbox'][2]) / 2)
        sorted_rows.append(row_dets)
    
    # Sort rows by average y-coordinate (top to bottom)
    sorted_rows.sort(key=lambda row: np.mean([(d['bbox'][1] + d['bbox'][3]) / 2 for d in row]))
    
    return sorted_rows


def fit_hash_row_line(hash_row: List[Dict]) -> Tuple[np.ndarray, np.ndarray]:
    """
    Fit a line through hash marks in a row using linear regression.
    
    Returns:
        (point_on_line, direction_vector)
    """
    points = []
    for det in hash_row:
        bbox = det['bbox']
        center_x = (bbox[0] + bbox[2]) / 2
        center_y = (bbox[1] + bbox[3]) / 2
        points.append([center_x, center_y])
    
    points = np.array(points)
    
    # Fit line using SVD
    mean_point = points.mean(axis=0)
    centered = points - mean_point
    _, _, vh = np.linalg.svd(centered)
    direction = vh[0]  # First principal component
    
    return mean_point, direction


# ============================================================================
# STEP 3: CONVERT PLAYER POSITIONS TO FIELD COORDINATES
# ============================================================================

class FieldCoordinateTransformer:
    """
    Transforms pixel coordinates to field coordinates using baseline and hash rows.
    """
    
    # NFL field constants
    FIELD_LENGTH = 100  # yards (0-100 scale)
    FIELD_WIDTH = 53.3  # yards
    NUMBERS_FROM_SIDELINE = 12  # yards
    HASH_SPACING = 1.0  # yards between hash rows
    
    def __init__(self, 
                 baseline_yard: int,
                 baseline_left: np.ndarray,
                 baseline_right: np.ndarray,
                 hash_rows: List[List[Dict]]):
        """
        Initialize transformer with detected field markings.
        
        Args:
            baseline_yard: Yard line value (e.g., 40)
            baseline_left: Left yard number center point [x, y]
            baseline_right: Right yard number center point [x, y]
            hash_rows: List of hash row detections
        """
        self.baseline_yard = baseline_yard
        self.baseline_left = baseline_left
        self.baseline_right = baseline_right
        
        # Baseline direction (across field, left to right)
        self.baseline_direction = baseline_right - baseline_left
        self.baseline_direction = self.baseline_direction / np.linalg.norm(self.baseline_direction)
        
        # Down-field direction (perpendicular to baseline)
        self.downfield_direction = np.array([-self.baseline_direction[1], self.baseline_direction[0]])
        
        # Process hash rows
        self.hash_rows = hash_rows
        self.hash_row_lines = []
        for row in hash_rows:
            if len(row) >= 2:
                point, direction = fit_hash_row_line(row)
                self.hash_row_lines.append((point, direction))
    
    def pixel_to_field(self, pixel_x: float, pixel_y: float) -> Tuple[float, float]:
        """
        Convert pixel coordinates to field coordinates.
        
        Args:
            pixel_x, pixel_y: Pixel coordinates
        
        Returns:
            (field_x, field_y) where:
                field_x: yards along field (0-100 scale)
                field_y: yards across field (0-53.3, 0=bottom sideline)
        """
        pixel_point = np.array([pixel_x, pixel_y])
        
        # === FIELD_X: Distance from baseline along downfield direction ===
        # Project onto downfield direction
        to_point = pixel_point - self.baseline_left
        distance_along_baseline = np.dot(to_point, self.baseline_direction)
        distance_downfield = np.dot(to_point, self.downfield_direction)
        
        # Convert pixels to yards using hash spacing as reference
        # Estimate pixels per yard from hash rows
        if len(self.hash_row_lines) >= 2:
            # Distance between hash rows in pixels
            row1_point = self.hash_row_lines[0][0]
            row2_point = self.hash_row_lines[1][0]
            pixel_distance = np.linalg.norm(row2_point - row1_point)
            pixels_per_yard = pixel_distance / self.HASH_SPACING
        else:
            # Fallback: estimate from baseline width
            baseline_width_pixels = np.linalg.norm(self.baseline_right - self.baseline_left)
            # Numbers are 12 yards from each sideline, so they span ~29.3 yards
            pixels_per_yard = baseline_width_pixels / 29.3
        
        # Convert downfield distance to yards
        distance_downfield_yards = distance_downfield / pixels_per_yard
        
        # Field_x relative to baseline
        field_x = self.baseline_yard + distance_downfield_yards
        
        # === FIELD_Y: Distance across field (sideline to sideline) ===
        # Use hash rows to determine y-position
        if self.hash_row_lines:
            # Find closest hash row
            distances_to_rows = []
            for row_point, row_dir in self.hash_row_lines:
                to_row = pixel_point - row_point
                # Distance perpendicular to row
                perp_dist = abs(np.dot(to_row, np.array([-row_dir[1], row_dir[0]])))
                distances_to_rows.append(perp_dist)
            
            # Interpolate based on position relative to hash rows
            # For simplicity, use linear interpolation between known positions
            # Hash marks are typically at 18.5 and 34.8 yards from bottom sideline
            
            # Assume hash rows are evenly distributed across visible field
            num_rows = len(self.hash_row_lines)
            row_positions_y = np.linspace(12, 53.3 - 12, num_rows)  # Between numbers
            
            # Find which row the point is nearest to
            row_y_positions = [line[0][1] for line in self.hash_row_lines]
            row_y_positions = np.array(row_y_positions)
            
            # Interpolate field_y based on pixel_y position among rows
            if num_rows >= 2:
                # Linear mapping from pixel space to field space
                pixel_y_min = row_y_positions.min()
                pixel_y_max = row_y_positions.max()
                field_y_min = row_positions_y[0]
                field_y_max = row_positions_y[-1]
                
                # Linear interpolation
                if pixel_y_max > pixel_y_min:
                    t = (pixel_y - pixel_y_min) / (pixel_y_max - pixel_y_min)
                    field_y = field_y_min + t * (field_y_max - field_y_min)
                else:
                    field_y = (field_y_min + field_y_max) / 2
            else:
                # Single row, assume it's at field center
                field_y = self.FIELD_WIDTH / 2
        else:
            # No hash rows, use baseline for estimate
            # Map across baseline width
            baseline_width_pixels = np.linalg.norm(self.baseline_right - self.baseline_left)
            t = distance_along_baseline / baseline_width_pixels
            # Numbers span from 12 to 41.3 yards (53.3 - 12)
            field_y = 12 + t * 29.3
        
        # Clamp to field boundaries
        field_x = np.clip(field_x, 0, self.FIELD_LENGTH)
        field_y = np.clip(field_y, 0, self.FIELD_WIDTH)
        
        return field_x, field_y
    
    def transform_players(self, player_detections: List[Dict]) -> List[Dict]:
        """
        Transform all player detections to field coordinates.
        
        Args:
            player_detections: List of player detection dicts with 'bbox' and 'class_name'
        
        Returns:
            List of dicts with added 'field_x' and 'field_y' keys
        """
        transformed = []
        for det in player_detections:
            bbox = det['bbox']
            # Use bottom-center of bounding box (player's feet)
            pixel_x = (bbox[0] + bbox[2]) / 2
            pixel_y = bbox[3]  # Bottom of box
            
            field_x, field_y = self.pixel_to_field(pixel_x, pixel_y)
            
            transformed.append({
                **det,
                'field_x': field_x,
                'field_y': field_y
            })
        
        return transformed


# ============================================================================
# STEP 4: VISUALIZATION
# ============================================================================

def plot_field_coordinates(player_data: List[Dict], 
                           baseline_yard: int,
                           title: str = "Top-Down Field View"):
    """
    Plot players on a top-down field visualization.
    
    Args:
        player_data: List of player dicts with 'field_x', 'field_y', 'class_name'
        baseline_yard: Line of scrimmage yard line
        title: Plot title
    """
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # Draw field boundaries
    ax.plot([0, 100], [0, 0], 'k-', linewidth=2)  # Bottom sideline
    ax.plot([0, 100], [53.3, 53.3], 'k-', linewidth=2)  # Top sideline
    ax.plot([0, 0], [0, 53.3], 'k-', linewidth=2)  # Left endline
    ax.plot([100, 100], [0, 53.3], 'k-', linewidth=2)  # Right endline
    
    # Draw yard lines (every 10 yards)
    for yard in range(10, 100, 10):
        ax.plot([yard, yard], [0, 53.3], 'k--', alpha=0.3, linewidth=1)
        ax.text(yard, -2, str(yard), ha='center', fontsize=10)
    
    # Draw hash marks (approximately)
    for yard in range(0, 101, 1):
        ax.plot([yard, yard], [18.5, 18.5], 'k-', linewidth=1, alpha=0.5)
        ax.plot([yard, yard], [34.8, 34.8], 'k-', linewidth=1, alpha=0.5)
    
    # Draw baseline (line of scrimmage) in bold
    ax.axvline(baseline_yard, color='red', linewidth=3, label=f'Line of Scrimmage ({baseline_yard} yd)')
    
    # Plot players
    offense_colors = {'offense': 'blue', 'qb': 'darkblue', 'rb': 'cyan'}
    defense_colors = {'defense': 'red', 'dl': 'darkred', 'lb': 'orange', 'db': 'pink'}
    
    for player in player_data:
        x = player['field_x']
        y = player['field_y']
        class_name = player.get('class_name', 'unknown').lower()
        
        # Determine color based on class
        if any(off in class_name for off in ['offense', 'qb', 'rb', 'wr', 'te', 'ol']):
            color = offense_colors.get(class_name, 'blue')
            marker = 'o'
        elif any(def_ in class_name for def_ in ['defense', 'dl', 'lb', 'db']):
            color = defense_colors.get(class_name, 'red')
            marker = 's'
        else:
            color = 'gray'
            marker = 'x'
        
        ax.scatter(x, y, c=color, s=100, marker=marker, edgecolors='black', linewidth=1.5)
    
    # Labels and formatting
    ax.set_xlim(-5, 105)
    ax.set_ylim(-5, 58.3)
    ax.set_xlabel('Yard Line', fontsize=12)
    ax.set_ylabel('Field Width (yards)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    return fig


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def transform_frame_to_field_coordinates(field_detections: List[Dict],
                                         player_detections: List[Dict],
                                         visualize: bool = True) -> Dict:
    """
    Complete pipeline to transform a frame's detections to field coordinates.
    
    Args:
        field_detections: Field marking detections (yard numbers, hashes)
        player_detections: Player detections
        visualize: Whether to generate visualization
    
    Returns:
        Dict with:
            - 'baseline_yard': int
            - 'players': List of player dicts with field coordinates
            - 'figure': matplotlib figure (if visualize=True)
    """
    # Separate field detections into yard numbers and hashes
    yard_detections = [d for d in field_detections if 'yard' in d.get('class_name', '').lower()]
    hash_detections = [d for d in field_detections if 'hash' in d.get('class_name', '').lower()]
    
    # STEP 1: Find baseline
    baseline_result = find_baseline_yard_line(yard_detections)
    if baseline_result is None:
        raise ValueError("Could not find a valid baseline yard line. Need at least 2 matching yard numbers.")
    
    baseline_yard, (left_det, right_det) = baseline_result
    baseline_left, baseline_right = get_baseline_points(left_det, right_det)
    
    print(f"✓ Baseline identified: {baseline_yard} yard line")
    
    # STEP 2: Identify hash rows
    hash_rows = cluster_hashes_into_rows(hash_detections)
    print(f"✓ Found {len(hash_rows)} hash rows")
    
    # STEP 3: Transform player positions
    transformer = FieldCoordinateTransformer(
        baseline_yard=baseline_yard,
        baseline_left=baseline_left,
        baseline_right=baseline_right,
        hash_rows=hash_rows
    )
    
    transformed_players = transformer.transform_players(player_detections)
    print(f"✓ Transformed {len(transformed_players)} players to field coordinates")
    
    # STEP 4: Visualize
    result = {
        'baseline_yard': baseline_yard,
        'players': transformed_players,
        'transformer': transformer
    }
    
    if visualize:
        fig = plot_field_coordinates(transformed_players, baseline_yard)
        result['figure'] = fig
    
    return result


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    # Example mock data for testing
    field_detections = [
        {'class_name': 'yard_40', 'confidence': 0.95, 'bbox': (100, 200, 150, 250), 
         'yard_number_pred': 40, 'yard_number_conf': 0.95},
        {'class_name': 'yard_40', 'confidence': 0.92, 'bbox': (850, 210, 900, 260), 
         'yard_number_pred': 40, 'yard_number_conf': 0.92},
        {'class_name': 'hash', 'confidence': 0.88, 'bbox': (200, 300, 220, 320)},
        {'class_name': 'hash', 'confidence': 0.87, 'bbox': (400, 305, 420, 325)},
        {'class_name': 'hash', 'confidence': 0.86, 'bbox': (600, 302, 620, 322)},
        {'class_name': 'hash', 'confidence': 0.89, 'bbox': (200, 450, 220, 470)},
        {'class_name': 'hash', 'confidence': 0.88, 'bbox': (400, 455, 420, 475)},
    ]
    
    player_detections = [
        {'class_name': 'qb', 'confidence': 0.98, 'bbox': (480, 350, 520, 420)},
        {'class_name': 'offense', 'confidence': 0.95, 'bbox': (450, 380, 490, 450)},
        {'class_name': 'defense', 'confidence': 0.93, 'bbox': (500, 280, 540, 340)},
        {'class_name': 'defense', 'confidence': 0.91, 'bbox': (520, 290, 560, 350)},
    ]
    
    result = transform_frame_to_field_coordinates(field_detections, player_detections)
    print("\nTransformed player positions:")
    for p in result['players']:
        print(f"  {p['class_name']}: ({p['field_x']:.1f}, {p['field_y']:.1f})")